In [ ]:
import pandas as pd
import numpy as np
%matplotlib inline

df = pd.read_csv('/content/drive/MyDrive/')
df = df.iloc[:, -2]
df = pd.DataFrame(df)
df

In [ ]:
end_date = pd.to_datetime('2023-07-18')
start_date = end_date - pd.Timedelta(days=369)

date_range = pd.date_range(start=start_date, periods=len(df))

df['Date'] = date_range
df['Date'] = pd.to_datetime(df['Date'])
df

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(0)
df = pd.DataFrame({'Date': df['Date'], 'BiLSTM': df['BiLSTM']})

df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date')

initial_cash = 100000

def backtest_strategy(df, initial_cash=100000, holding_ratio=0.4, transaction_cost=0.003, slippage=0.002, change_threshold=0.02):
    cash = initial_cash
    position = 0
    total_turnover = 0
    returns = []

    df['Short_MA'] = df['BiLSTM'].rolling(window=5).mean()
    df['Long_MA'] = df['BiLSTM'].rolling(window=40).mean()
    df = df.dropna()

    df_monthly = df.resample('M').last()

    for i in range(1, len(df_monthly)):
        prev_short_ma = df_monthly['Short_MA'].iloc[i - 1]
        prev_long_ma = df_monthly['Long_MA'].iloc[i - 1]
        short_ma = df_monthly['Short_MA'].iloc[i]
        long_ma = df_monthly['Long_MA'].iloc[i]
        current_price = df_monthly['BiLSTM'].iloc[i]

        if prev_short_ma <= prev_long_ma and short_ma > long_ma and (short_ma - long_ma) / long_ma > change_threshold:
            num_shares = int((cash * holding_ratio) // current_price)
            if num_shares > 0:
                cost = num_shares * current_price * (1 + transaction_cost + slippage)
                cash -= cost
                position += num_shares
                total_turnover += num_shares
                print(f"Buy {num_shares} shares at {current_price:.2f}")

        elif prev_short_ma >= prev_long_ma and short_ma < long_ma and (long_ma - short_ma) / long_ma > change_threshold:
            if position > 0:
                num_shares = int(position * holding_ratio)
                revenue = num_shares * current_price * (1 - transaction_cost - slippage)
                cash += revenue
                position -= num_shares
                total_turnover += num_shares
                print(f"Sell {num_shares} shares at {current_price:.2f}")

        portfolio_value = cash + position * current_price
        returns.append(portfolio_value)

    return pd.Series(returns, index=df_monthly.index[1:]), total_turnover, cash + position * df_monthly['BiLSTM'].iloc[-1]

portfolio_returns, total_turnover, final_portfolio_value = backtest_strategy(df)

total_return = (final_portfolio_value - initial_cash) / initial_cash
monthly_returns = portfolio_returns.pct_change().dropna()
annualized_sharpe_ratio = monthly_returns.mean() / monthly_returns.std() * np.sqrt(12) if monthly_returns.std() != 0 else 0
cagr = (final_portfolio_value / initial_cash) ** (1 / (len(df) / 252)) - 1
rolling_max = portfolio_returns.cummax()
drawdowns = (portfolio_returns - rolling_max) / rolling_max
max_drawdown = drawdowns.min()
calmar_ratio = cagr / abs(max_drawdown) if max_drawdown != 0 else np.nan

print(f"Turnover Ratio: {total_turnover / len(df):.2f}")
print(f"Final Portfolio Value: {final_portfolio_value:.2f}")
print(f"Total Return: {total_return:.2%}")
print(f"Sharpe Ratio (Annualized): {annualized_sharpe_ratio:.2f}")
print(f"CAGR: {cagr:.2%}")
print(f"Maximum Drawdown: {max_drawdown:.2%}")
print(f"Calmar Ratio: {calmar_ratio:.2f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="muted")

plt.figure(figsize=(14, 7))

plt.plot(portfolio_returns.index, portfolio_returns, label="Portfolio Value", color='royalblue', linewidth=2)

plt.title("Portfolio Value Over Time", fontsize=16, fontweight='bold')
plt.xlabel("Date", fontsize=14)
plt.ylabel("Portfolio Value", fontsize=14)

plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(loc='upper left', fontsize=12)


max_value_index = portfolio_returns.idxmax()
max_value = portfolio_returns.max()
plt.annotate(f"Max Value: {max_value:.2f}",
             xy=(max_value_index, max_value),
             xytext=(max_value_index, max_value + 0.05),
             arrowprops=dict(facecolor='black', arrowstyle="->"),
             fontsize=12, color='black')


min_value_index = portfolio_returns.idxmin()
min_value = portfolio_returns.min()
plt.annotate(f"Min Value: {min_value:.2f}",
             xy=(min_value_index, min_value),
             xytext=(min_value_index, min_value - 0.05),
             arrowprops=dict(facecolor='red', arrowstyle="->"),
             fontsize=12, color='red')


plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set(style="whitegrid", palette="muted")

plt.figure(figsize=(12, 6))

plt.hist(daily_returns, bins=50, alpha=0.7, color='skyblue', edgecolor='black')

mean_return = np.mean(daily_returns)
median_return = np.median(daily_returns)

plt.axvline(mean_return, color='red', linestyle='dashed', linewidth=2, label=f"Mean: {mean_return:.4f}")
plt.axvline(median_return, color='green', linestyle='dashed', linewidth=2, label=f"Median: {median_return:.4f}")

plt.title("Histogram of Daily Returns", fontsize=16, fontweight='bold')
plt.xlabel("Daily Return", fontsize=14)
plt.ylabel("Frequency", fontsize=14)

plt.legend(loc='upper left', fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="muted")

cumulative_returns = (1 + daily_returns).cumprod() - 1

plt.figure(figsize=(14, 7))

plt.plot(daily_returns.index, cumulative_returns, label="Cumulative Returns", color='green', linewidth=2)

plt.title("Cumulative Returns Over Time", fontsize=16, fontweight='bold')
plt.xlabel("Date", fontsize=14)
plt.ylabel("Cumulative Returns", fontsize=14)

plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(loc='upper left', fontsize=12)

plt.fill_between(daily_returns.index, 0, cumulative_returns, where=(cumulative_returns > 0),
                 facecolor='green', alpha=0.3, interpolate=True)

max_return_index = cumulative_returns.idxmax()
max_return_value = cumulative_returns.max()
plt.annotate(f"Max Return: {max_return_value:.2%}",
             xy=(max_return_index, max_return_value),
             xytext=(max_return_index, max_return_value + 0.05),
             arrowprops=dict(facecolor='black', arrowstyle="->"),
             fontsize=12, color='black')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set(style="whitegrid", palette="muted")

plt.figure(figsize=(14, 7))

rolling_max = portfolio_returns.cummax()

plt.plot(portfolio_returns.index, portfolio_returns, label="Portfolio Value", color='royalblue', linewidth=2)

plt.fill_between(portfolio_returns.index, portfolio_returns, rolling_max, color='red', alpha=0.3, label="Drawdown")

max_drawdown_index = portfolio_returns.idxmin()
max_drawdown_value = portfolio_returns.min()
plt.annotate(f"Max Drawdown: {max_drawdown_value:.2f}",
             xy=(max_drawdown_index, max_drawdown_value),
             xytext=(max_drawdown_index, max_drawdown_value - 0.05),
             arrowprops=dict(facecolor='black', arrowstyle="->"),
             fontsize=12, color='black')

plt.title("Portfolio Value and Drawdowns", fontsize=16, fontweight='bold')
plt.xlabel("Date", fontsize=14)
plt.ylabel("Portfolio Value", fontsize=14)

plt.grid(True, linestyle='--', alpha=0.7)

plt.legend(loc='upper left', fontsize=12)

plt.tight_layout()

plt.show()


In [ ]:
initial_cash = 100000
cash = initial_cash
position = 0
total_turnover = 0
transaction_cost = 0.001
slippage = 0.001
returns = []


def backtest_strategy(df, initial_cash=100000, holding_ratio=0.5, transaction_cost=0.001, slippage=0.001):
    cash = initial_cash
    position = 0
    total_turnover = 0
    returns = []

    for i in range(1, len(df)):
        prev_price = df['ATFN'].iloc[i - 1]
        current_price = df['ATFN'].iloc[i]

        if current_price > prev_price:
            if cash > current_price:
                num_shares = int((cash * holding_ratio) // current_price)
                cost = num_shares * current_price * (1 + transaction_cost + slippage)
                cash -= cost
                position += num_shares
                total_turnover += num_shares
                print(f"Buy {num_shares} shares at {current_price * (1 + slippage):.2f}")
        elif current_price < prev_price:
            if position > 0:
                num_shares = int(position * holding_ratio)
                revenue = num_shares * current_price * (1 - transaction_cost - slippage)
                cash += revenue
                total_turnover += num_shares
                position -= num_shares
                print(f"Sell {num_shares} shares at {current_price * (1 - slippage):.2f}")

        portfolio_value = cash + position * current_price
        returns.append(portfolio_value)

    return pd.Series(returns), total_turnover, cash + position * df['ATFN'].iloc[-1]

portfolio_returns, total_turnover, final_portfolio_value = backtest_strategy(df)

total_return = (final_portfolio_value - initial_cash) / initial_cash
daily_returns = portfolio_returns.pct_change().dropna()
sharpe_ratio = daily_returns.mean() / daily_returns.std() * np.sqrt(252) if daily_returns.std() != 0 else 0
turnover_ratio = total_turnover / len(df)

print(f"Final Portfolio Value: {final_portfolio_value:.2f}")
print(f"Total Return: {total_return:.2%}")
print(f"Total Turnover: {total_turnover}")
print(f"Turnover Ratio: {turnover_ratio:.2%}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")